# 04 · Deep Learning Fundamentals

Use PyTorch and TensorFlow/Keras to study gradients, MLP/autoencoder/recurrent models, activation and normalization choices, optimizers, schedulers, class weighting, early stopping, and overfitting.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Trace forward pass, loss, backpropagation, optimizer step, and scheduler step.
- Compare five activations, two initialization families, four normalization choices, and three optimizers.
- Read training/validation curves instead of reporting only final accuracy.
- Compare RNN/LSTM/GRU under controlled data and epochs.
- Use reconstruction error for anomaly ranking and threshold decisions.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

## Autograd in one cell

For $y=(wx+b)^2$, PyTorch builds a dynamic graph and accumulates gradients after `backward()`.
Optimizers then mutate parameters; `zero_grad()` prevents unintended accumulation.

In [ ]:
import torch

x = torch.tensor([2.0])
weight = torch.tensor([3.0], requires_grad=True)
bias = torch.tensor([1.0], requires_grad=True)
loss = (weight * x + bias).pow(2).mean()
loss.backward()
print({"loss": loss.item(), "d_loss_d_weight": weight.grad.item(), "d_loss_d_bias": bias.grad.item()})

In [ ]:
metric_path = METRICS_DIR / "deep.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_deep.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "deep.json")], check=True)
result = json.loads(metric_path.read_text())
print("config", Path("configs/models/deep.yaml").read_text())
print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
import pandas as pd

ablations = result["pytorch"]["tabular_mlp_ablations"]
activation_table = pd.DataFrame(ablations["activations"]).T
normalization_table = pd.DataFrame(ablations["normalization"]).T
optimizer_table = pd.DataFrame(ablations["optimizers"]).T
display(activation_table[["roc_auc", "best_validation_loss", "overfitting_gap"]])
display(normalization_table[["roc_auc", "best_validation_loss", "overfitting_gap"]])
display(optimizer_table[["roc_auc", "best_validation_loss", "overfitting_gap"]])

## Activations and initialization

ReLU is cheap but can die; LeakyReLU retains a negative slope; GELU gates smoothly; sigmoid/tanh
saturate. Kaiming initialization matches rectifiers; Xavier targets stable variance for symmetric
activations. Short synthetic runs may rank them noisily—inspect multiple seeds before generalizing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

values = torch.linspace(-5, 5, 300)
functions = {
    "relu": torch.relu(values), "leaky_relu": torch.nn.functional.leaky_relu(values),
    "gelu": torch.nn.functional.gelu(values), "sigmoid": torch.sigmoid(values), "tanh": torch.tanh(values),
}
for name, output in functions.items():
    plt.plot(values, output, label=name)
plt.legend(); plt.title("Activation functions"); plt.show()

## Normalization

BatchNorm uses batch statistics, LayerNorm normalizes within each example, and GroupNorm divides
channels/features into groups. Small batches can make BatchNorm noisy; tabular GroupNorm here is an
educational parity exercise rather than a default recommendation.

In [ ]:
normalization_table["roc_auc"].plot.bar(title="Normalization ablation")
plt.ylim(0, 1)
plt.show()

## Optimizers

SGD with momentum has simple dynamics, Adam adapts first/second moments, and RMSProp scales by a
moving squared-gradient average. A fair comparison should tune learning rate per optimizer; this
controlled quick run intentionally holds most other choices fixed.

In [ ]:
optimizer_table[["roc_auc", "best_validation_loss"]].plot.bar(subplots=True, figsize=(9, 6), title=["ROC-AUC", "Validation loss"])
plt.tight_layout(); plt.show()

## Learning-rate schedules

Warmup gradually increases the learning rate, StepLR drops it discretely, and cosine annealing
decays smoothly. The recorded trace proves scheduler behavior rather than only naming it.

In [ ]:
schedulers = ablations["schedulers"]
for name, values in schedulers.items():
    plt.plot(values["learning_rates"], marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.legend(); plt.show()

## Early stopping and overfitting

The best validation checkpoint is restored. `overfitting_gap = final validation loss - final train
loss`; a large positive gap is a warning, not a proof. Curves and multiple seeds are stronger evidence.

In [ ]:
chosen = ablations["class_weighting"]
plt.plot(chosen["train_loss"], marker="o", label="train")
plt.plot(chosen["validation_loss"], marker="o", label="validation")
plt.title(f"Class-weighted MLP; gap={chosen['overfitting_gap']:.3f}")
plt.legend(); plt.show()

## Class weighting

Inverse-frequency weights increase loss contribution from rare churners. They can improve ranking or
recall while reducing threshold accuracy and making optimization less stable. Tune the decision
threshold after training rather than assuming 0.5 is correct.

In [ ]:
print("class-weighted", {key: chosen[key] for key in ("accuracy", "roc_auc", "class_weighted", "overfitting_gap")})

## Sequence models

RNN repeatedly transforms hidden state; LSTM adds input/forget/output gates and a cell state; GRU
uses update/reset gates with fewer parameters. The synthetic target depends on recent amount trend.

In [ ]:
pytorch_sequence = pd.DataFrame(result["pytorch"]["sequence_models"]).T
keras_sequence = pd.DataFrame(result["tensorflow"]["sequence_models"]).T
display(pytorch_sequence[["accuracy", "roc_auc", "overfitting_gap"]])
display(keras_sequence[["roc_auc", "epochs_ran", "parameters"]])

## Framework parity

PyTorch and Keras use the same generated sequences but not identical initialization, architecture,
batch order, or optimizer implementation. Compare workflow and trends; do not treat score differences
as a framework benchmark.

In [ ]:
print("Keras tabular MLP", result["tensorflow"]["tabular_mlp"])

## Autoencoder anomaly detection

The autoencoder learns normal transaction reconstruction. Reconstruction error ranks anomalies;
the 99th percentile of normal error creates an explicit threshold with precision/recall tradeoff.

In [ ]:
anomaly = result["pytorch"]["autoencoder"]
display(pd.Series(anomaly).drop("training_loss"))
plt.plot(anomaly["training_loss"], marker="o"); plt.title("Autoencoder epoch loss"); plt.show()

## TensorBoard

Run `tensorboard --logdir artifacts/tensorboard` locally. TensorBoard is useful for synchronized
curves and comparisons, but event files do not replace versioned configs and JSON summaries.

## Exercises and interview checkpoint

1. Repeat every ablation across five seeds and add confidence intervals.
2. Tune optimizer-specific learning rates.
3. Plot threshold precision/recall for autoencoder error.
4. Create an intentionally overfit MLP and diagnose it from curves.
5. Explain why the quick-profile Keras scores are not evidence that one framework is superior.